In [1]:
import configparser
import json
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

W1122 15:49:12.813000 17068 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
# Если искать по всем частям речи (результаты не очень; тот, что ниже, показывает более хорошие результаты)
class KeywordExtractor:
    """Инициализация экстрактора ключевых слов.
    param model_name: Название предобученной модели SentenceTransformer
    param chunk_size: Максимальное количество слов в одном фрагменте (чанке) текста
    param overlap: Количество перекрывающихся слов между соседними чанками
    param top_n: Итоговое количество лучших ключевых слов, возвращаемых для одной статьи
    """
    def __init__(self, model_name: str, chunk_size: int, overlap: int, top_n: int):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.top_n = top_n

        # Инициализируем модель эмбеддингов и передаем её в KeyBERT
        model = SentenceTransformer(model_name)
        self.kw_model = KeyBERT(model)

    def split_text(self, text: str) -> list:
        """Разбиваем длинный текст на фрагменты с заданным перекрытием"""
        words = text.split()
        if not words:
            return []
        step = self.chunk_size - self.overlap
        return [" ".join(words[i : i + self.chunk_size]) for i in range(0, len(words), step)]

    def extract_from_article(self, text: str) -> list:
        doc = text.replace("\n", " ")
        chunks = self.split_text(doc)
        if not chunks:
            return []

        # Ищем одиночные слова и биграммы, исключаем стоп-слова, ставим разнообразие 0.4, берем топ-10 кандидатов
        keywords = self.kw_model.extract_keywords(chunks, keyphrase_ngram_range=(1, 2), stop_words="english",
                                                 use_mmr=True, diversity=0.4, top_n=10)
        all_kws = {}
        for kw_list in keywords:
            for word, score in kw_list:
                if word not in all_kws or score > all_kws[word]:
                    all_kws[word] = round(score, 4)

        sorted_kws = sorted(all_kws.items(), key=lambda x: x[1], reverse=True)
        return sorted_kws[: self.top_n]
  
    def process_dataset(self, data: dict) -> dict:
        for category, articles in data.items():
            for article in tqdm(articles, desc=f"Обработка {category}"):
                article["keywords"] = self.extract_from_article(article["text"])
        return data

In [ ]:
# Если ищем либо существительные, либо прил+сущ
import nltk
from sklearn.feature_extraction.text import CountVectorizer

nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("punkt_tab", quiet=True)

def extract_noun_phrases(text: str) -> list:
    words = [w.lower() for w in nltk.word_tokenize(text) if w.isalpha() and len(w) >= 3]
    tagged = nltk.pos_tag(words)

    candidates = []
    for i in range(len(tagged)):
        word, tag = tagged[i]
        # Одиночное существительное
        if tag.startswith("NN"):
            candidates.append(word)

        # Прилагательное + существительное
        if i < len(tagged) - 1:
            next_word, next_tag = tagged[i + 1]
            if tag.startswith("JJ") and next_tag.startswith("NN"):
                candidates.append(f"{word} {next_word}")

    return candidates

class KeywordExtractor:
    def __init__(self, model_name: str, chunk_size: int, overlap: int, top_n: int):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.top_n = top_n

        model = SentenceTransformer(model_name)
        self.kw_model = KeyBERT(model)
        self.vectorizer = CountVectorizer(analyzer=extract_noun_phrases)

    def split_text(self, text: str) -> list:
        words = text.split()
        if not words:
            return []
        step = self.chunk_size - self.overlap
        return [" ".join(words[i : i + self.chunk_size]) for i in range(0, len(words), step)]

    def extract_from_article(self, text: str) -> list:
        doc = text.replace("\n", " ")
        chunks = self.split_text(doc)
        if not chunks:
            return []

        keywords = self.kw_model.extract_keywords(chunks, vectorizer=self.vectorizer, use_mmr=True, diversity=0.4, top_n=10)

        all_kws = {}
        for kw_list in keywords:
            for word, score in kw_list:
                if word not in all_kws or score > all_kws[word]:
                    all_kws[word] = round(score, 4)

        sorted_kws = sorted(all_kws.items(), key=lambda x: x[1], reverse=True)
        return sorted_kws[: self.top_n]

    def process_dataset(self, data: dict) -> dict:
        for category, articles in data.items():
            for article in tqdm(articles, desc=f"Обработка {category}"):
                article["keywords"] = self.extract_from_article(article["text"])
        return data

In [3]:
config = configparser.ConfigParser()
config.read("config.ini", encoding="utf-8")

input_file = config['KEYWORDS']['input_file']
output_file = config['KEYWORDS']['output_file']
model_name = config['KEYWORDS']['model_name']
chunk_size = config.getint('KEYWORDS', 'chunk_size')
overlap = config.getint('KEYWORDS', 'overlap')
top_n = config.getint('KEYWORDS', 'top_n')

with open(input_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

extractor = KeywordExtractor(model_name=model_name, chunk_size=chunk_size, overlap=overlap, top_n=top_n)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# проверка на первой статье по всем частям речи (много бессмысленных, например, aorta underwent, aorta patients, fenestrated extracardiac)
first_article = raw_data["medicine"][0]
print(extractor.extract_from_article(first_article["text"]))

[('aortic dissection', 0.6332), ('reconstruction aortic', 0.6321), ('bicuspid aortic', 0.6224), ('ascending aorta', 0.6201), ('aortic root', 0.6095), ('aorta dilation', 0.6038), ('aorta underwent', 0.5903), ('aortic arch', 0.5902), ('neo aortic', 0.5841), ('fenestrated extracardiac', 0.5828), ('aorta patients', 0.5799), ('aortic valves', 0.5777), ('aorta dissection', 0.5722), ('congenital heart', 0.5509), ('aorta height', 0.5356)]


In [ ]:
# проверка на первой статье по сущ/прил+сущ
first_article = raw_data["medicine"][0]
print(extractor.extract_from_article(first_article["text"]))

[('aortic dissection', 0.6332), ('aortic root', 0.6095), ('aorta dilation', 0.6038), ('aortic arch', 0.5902), ('aortic valves', 0.5777), ('aorta dissection', 0.5722), ('congenital heart', 0.5509), ('aorta replacement', 0.5429), ('extracardiac fontan', 0.5225), ('single ventricle', 0.5217), ('right ventricle', 0.5002), ('underwent valve', 0.4916), ('norwood operation', 0.4871), ('atrioventricular valve', 0.4633), ('cavopulmonary anastomosis', 0.4229)]


In [ ]:
processed_data = extractor.process_dataset(raw_data)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(processed_data, f, ensure_ascii=False, indent=4)